In [0]:
%sql
SHOW TABLES FROM alpenia_dev.bronze;

In [0]:
%sql
SELECT DISTINCT status
FROM alpenia_dev.bronze.brz_contracts;

SELECT DISTINCT action
FROM alpenia_dev.bronze.brz_web_events;

SELECT DISTINCT peril
FROM alpenia_dev.bronze.brz_claims;

SELECT DISTINCT finding_type
FROM alpenia_dev.bronze.brz_expertises;

# Question 1 — Contrats actifs par ville et par produit

Question : Combien de contrats actifs sont enregistrés par ville et par produit ?



In [0]:
%sql
SELECT
    city,
    product,
    COUNT(*) AS active_contracts
FROM alpenia_dev.bronze.brz_contracts
WHERE LOWER(status) = 'active'
GROUP BY city, product
ORDER BY city, product;

# Question 2 — Sinistres, montants réglés et ratio de sinistralité par ville
CELLULE MARKDOWN

Question : Quel est le nombre de sinistres, le montant total réglé et le ratio de sinistralité par ville ?

Pour éviter de multiplier artificiellement les primes lorsqu'un contrat possède plusieurs sinistres, on calcule séparément les agrégats des contrats et ceux des sinistres.



Le ratio est exprimé en pourcentage :
\[
\text{Ratio} =
\frac{\text{Montant des sinistres}}{\text{Primes annuelles}} \times 100
\]

Ici, on utilise amount_eur comme montant réglé, conformément aux colonnes disponibles. Si cette colonne représente un montant différent du montant effectivement réglé dans ton modèle, il faudra adapter le calcul.

## INTERPRÉTATION

« Le ratio de sinistralité permet de comparer les montants des sinistres aux primes annuelles collectées dans chaque ville. Un ratio élevé indique que les sinistres représentent une part importante des primes. »




In [0]:
%sql
WITH primes AS (
    SELECT
        city,
        SUM(annual_premium_eur) AS total_annual_premiums
    FROM alpenia_dev.bronze.brz_contracts
    GROUP BY city
),

sinistres AS (
    SELECT
        c.city,
        COUNT(*) AS claim_count,
        SUM(s.amount_eur) AS total_claim_amount,
        AVG(s.amount_eur) AS average_claim_amount
    FROM alpenia_dev.bronze.brz_claims s
    JOIN alpenia_dev.bronze.brz_contracts c
        ON s.contract_id = c.contract_id
    GROUP BY c.city
)

SELECT
    p.city,
    COALESCE(s.claim_count, 0) AS claim_count,
    ROUND(COALESCE(s.total_claim_amount, 0), 2) AS total_claim_amount,
    ROUND(s.average_claim_amount, 2) AS average_claim_amount,
    ROUND(
        100.0 * COALESCE(s.total_claim_amount, 0)
        / NULLIF(p.total_annual_premiums, 0),
        2
    ) AS loss_ratio_percent
FROM primes p
LEFT JOIN sinistres s
    ON p.city = s.city
ORDER BY loss_ratio_percent DESC;

# Question 3 — Délai moyen de règlement par nature de sinistre

Question : Quel est le délai moyen de règlement, en jours, par nature de sinistre ?

## INTERPRÉTATION

« Cette analyse permet de comparer les délais moyens de règlement selon la nature du sinistre et d'identifier les catégories nécessitant potentiellement un traitement plus long. »

In [0]:
%sql

SELECT
    peril,
    ROUND(AVG(settlement_days), 1) AS average_settlement_days,
    COUNT(*) AS claim_count
FROM alpenia_dev.bronze.brz_claims
WHERE settlement_days IS NOT NULL
GROUP BY peril
ORDER BY average_settlement_days DESC;

#Question 4 — Les 10 clients ayant abandonné le plus de devis

Question : Quels sont les 10 clients ayant abandonné le plus de devis dans les interactions web ?
À adapter aux valeurs réelles de action

Il faut d'abord identifier le libellé exact correspondant à un abandon dans le résultat de SELECT DISTINCT action.

Si la valeur est quote_abandoned, par exemple, la requête sera :

Ne lance pas cette version tant que nous n'avons pas confirmé le libellé réel de l'action.

## INTERPRÉTATION

« Cette analyse identifie les clients qui abandonnent le plus souvent leurs devis et permet de repérer des comportements pouvant justifier une action commerciale ciblée. »


In [0]:
%sql
SELECT
    client_id,
    COUNT(*) AS abandoned_quotes
FROM alpenia_dev.bronze.brz_web_events
WHERE LOWER(action) = 'quote_abandoned'
  AND client_id IS NOT NULL
GROUP BY client_id
ORDER BY abandoned_quotes DESC
LIMIT 10;

In [0]:
# Preuve que les conversions n'ont pas introduit de NULL

df_web = spark.table(table_name("brz_web_events"))

null_before = df_web.filter(
    F.col("event_ts").isNull()
).count()

df_web = df_web.withColumn(
    "event_ts",
    F.to_timestamp("event_ts", "yyyy-MM-dd'T'HH:mm:ssX")
)

null_after = df_web.filter(
    F.col("event_ts").isNull()
).count()

print("NULL avant :", null_before)
print("NULL après :", null_after)

# Question 5 — Coût des expertises et expertises ouvertes

Question : Quel est le coût total des expertises par type de constat, et combien d'expertises sont encore ouvertes ?


In [0]:
%sql
SELECT
    finding_type,
    SUM(cost_eur) AS total_expertise_cost,
    COUNT(*) AS expertise_count,
    SUM(
        CASE WHEN is_open THEN 1 ELSE 0 END
    ) AS open_expertises
FROM alpenia_dev.bronze.brz_expertises
GROUP BY finding_type
ORDER BY total_expertise_cost DESC;

# Examiner les métadonnées

In [0]:
%sql
DESCRIBE DETAIL alpenia_dev.bronze.brz_contracts;

In [0]:
%sql
SELECT contract_id, status
FROM alpenia_dev.bronze.brz_contracts
LIMIT 5;

In [0]:
%sql
UPDATE alpenia_dev.bronze.brz_contracts
SET status = 'TEST_MODIFICATION'
WHERE contract_id = 'CT-00001';

In [0]:
%sql
SELECT contract_id, status
FROM alpenia_dev.bronze.brz_contracts
WHERE contract_id = 'CT-00001';

In [0]:
%sql
DESCRIBE HISTORY alpenia_dev.bronze.brz_contracts;

In [0]:
%sql
RESTORE TABLE alpenia_dev.bronze.brz_contracts
TO VERSION AS OF  2;

In [0]:
%sql
DESCRIBE HISTORY alpenia_dev.bronze.brz_contracts;